# Smart IDS v3 — Notebook 01: Introduction & Security Basics
An Intelligent Network Intrusion Detection System for AI Engineers

---

### What You Will Learn
1. **The Intrusion Detection Problem**: Why traditional signature-based firewalls fail against modern cyber attacks.
2. **Security Concepts for AI Engineers**: Translating networking jargon (flows, 5-tuples, packets) into tabular ML representations.
3. **Attack Taxonomy**: Understanding the 8 attack families in the CICIDS2017 benchmark dataset.
4. **Data Ingestion**: Loading raw daily CSV files or generating a high-fidelity synthetic fallback.
5. **Exploratory Data Analysis (EDA)**: Investigating feature distributions, zero-division rate artifacts, and severe class imbalance.

---

### Prerequisites
- Python 3.11, pandas, matplotlib, and scikit-learn basics.
- **Zero prior knowledge of networking or cybersecurity is required!** All security concepts are introduced from first principles with ML analogies.


## 1. Security Concepts for AI Engineers

Before looking at data, let's establish a shared vocabulary between cybersecurity and machine learning.

> **Security Concept: What is a Network Flow?**
> In computer networking, data moves in chunks called **packets**. A **flow** is a bidirectional conversation between two computers (for example, your laptop requesting a web page from a web server). 
> 
> *Machine Learning Analogy:* A network flow is simply **one row in a tabular dataset**! Each column is a summary statistic computed over that conversation (such as duration, total bytes sent, or average packet size).

> **Security Concept: What is the 5-Tuple?**
> Every flow is uniquely identified by five network properties:
> 1. Source IP (who sent it)
> 2. Destination IP (who received it)
> 3. Source Port (the application port on the sender)
> 4. Destination Port (the target service, e.g., Port 80 for HTTP, Port 22 for SSH)
> 5. Protocol (TCP or UDP)

> **Security Concept: What is CICFlowMeter?**
> CICFlowMeter is a network feature-extraction tool developed by the Canadian Institute for Cybersecurity. It listens to raw network packet streams and calculates **76 numerical statistical features** for each flow.

> **Security Concept: Attack Taxonomy (The 8 Classes)**
> - **BENIGN**: Normal, safe network traffic (web browsing, file transfers, email).
> - **DoS Hulk**: Denial-of-Service attack using high-volume HTTP request flooding to overwhelm a web server.
> - **PortScan**: Reconnaissance attack where an adversary probes hundreds of ports to see which doors are open.
> - **DDoS**: Distributed Denial-of-Service attack where traffic is coordinated across thousands of compromised computers.
> - **DoS GoldenEye**: Denial-of-Service attack that slowly consumes server connection pools with keep-alive headers.
> - **FTP-Patator**: Automated dictionary brute-force password guessing against an FTP file server (Port 21).
> - **SSH-Patator**: Automated dictionary brute-force password guessing against an encrypted SSH shell (Port 22).
> - **Web Attack**: SQL injection or cross-site scripting (XSS) attempting to steal data or compromise a web app.


## 2. Setting Up the Environment and Reproducibility

Let's import our libraries, verify paths, and set a fixed seed (42) so that all results are 100% reproducible.


In [ ]:
import sys
from pathlib import Path

cur = Path.cwd()
root = cur if (cur / 'src').exists() else cur.parent
if str(root / 'src') not in sys.path:
    sys.path.insert(0, str(root / 'src'))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from smart_ids.config import (
    CLASS_NAMES,
    CLASS_WEIGHTS,
    DATA_DIR,
    PROCESSED_DATA_DIR,
    RAW_DATA_DIR,
    SEED,
)
from smart_ids.data_loader import load_cicids2017

np.random.seed(SEED)
pd.set_option('display.max_columns', 15)
print(f"Environment initialized. Seed fixed to {SEED}.")


## 3. Loading the Benchmark Dataset

We load the CICIDS2017 dataset using `load_cicids2017`. If the 8 daily CSV files are placed in `data/raw/`, they are merged and sampled. If they are missing, the pipeline automatically activates a schema-identical synthetic fallback generator.

**What we do and why:**
We load a representative sample so that students can run the entire pipeline smoothly.


In [ ]:
# Load dataset (or generate synthetic fallback if raw CSVs are not present)
# For fast execution and learning, sample ~25,000 rows; can be scaled to 200,000 for final production benchmark
df_raw = load_cicids2017(data_dir=RAW_DATA_DIR, sample_size=25000, random_state=SEED)
print(f"Dataset Loaded Successfully!")
print(f"Total Rows: {df_raw.shape[0]:,}")
print(f"Total Columns: {df_raw.shape[1]}")
df_raw.head(3)


## 4. Exploratory Data Analysis: Class Distribution

Now let's examine the class distribution of our network traffic.

> **Common Mistake: Naive Accuracy on Imbalanced Data**
> In real-world enterprise networks, over 80% to 99% of all traffic is normal (BENIGN). If a dataset is 83% BENIGN, a dummy model that always guesses "BENIGN" achieves **83% accuracy while detecting ZERO attacks**!
> 
> In cybersecurity, a **False Negative** (missing a real attack) is catastrophic: an adversary breaches your network undetected. Never use raw accuracy alone to evaluate an Intrusion Detection System.


In [ ]:
# Compute class distribution counts and percentages
class_counts = df_raw['Label'].value_counts()
class_pcts = df_raw['Label'].value_counts(normalize=True) * 100

dist_df = pd.DataFrame({'Flow Count': class_counts, 'Percentage (%)': class_pcts})
print("--- Class Distribution ---")
print(dist_df.round(2))

# Visualize class distribution
plt.figure(figsize=(10, 5))
sns.barplot(x=class_counts.values, y=class_counts.index, palette='viridis')
plt.title("Empirical Class Distribution in Network Telemetry", fontsize=13, fontweight='bold')
plt.xlabel("Number of Flows", fontsize=11)
plt.ylabel("Traffic Class", fontsize=11)
plt.grid(axis='x', linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()


**What did we learn?**
The data is severely imbalanced: BENIGN traffic dominates (~83%), while critical attacks like Web Attack, FTP-Patator, and SSH-Patator represent around 1% each. We must use techniques like SMOTE and weighted evaluation metrics to handle this asymmetry.


## 5. Investigating Data Quality Pathologies

Network flow features generated by CICFlowMeter suffer from a known data-quality pathology: **infinite values (`+inf`, `-inf`)**.

> **Security Concept: Why do infinite values appear in network data?**
> Rate features such as `Flow Bytes/s` and `Flow Packets/s` are calculated by dividing the total bytes by `Flow Duration`. If a flow starts and ends instantly (duration = 0 microseconds), the computer divides by zero, resulting in `+inf` or `-inf`.

Let's detect how many infinite and missing values exist in the raw data.


In [ ]:
# Count infinite values across numerical features
numeric_cols = df_raw.select_dtypes(include=[np.number]).columns
num_infs = int(np.isinf(df_raw[numeric_cols].to_numpy()).sum())
num_nans = int(df_raw[numeric_cols].isna().sum().sum())
num_dupes = int(df_raw.duplicated().sum())

print(f"Data Quality Diagnostics:")
print(f" - Infinite values detected: {num_infs:,}")
print(f" - Missing values (NaN) detected: {num_nans:,}")
print(f" - Exact duplicate rows detected: {num_dupes:,}")


## 6. Persisting Raw Dataset Sample for Downstream Notebooks

To ensure all subsequent notebooks (02 through 07) use the exact same data split, we save this raw sample to `data/processed/cicids2017_raw_sample.parquet`.


In [ ]:
# Cell 5 in Notebook 01
processed_raw_path = PROCESSED_DATA_DIR / 'cicids2017_raw_sample.csv'
processed_raw_path.parent.mkdir(parents=True, exist_ok=True)
df_raw.to_csv(processed_raw_path, index=False)
print(f"Saved dataset snapshot to: {processed_raw_path}")


---
## Summary and Next Notebook

In this notebook, we learned:
- How network packets are grouped into bidirectional **flows** (tabular rows).
- The 8 attack categories in the CICIDS2017 dataset.
- Why naive accuracy is misleading on heavily imbalanced network traffic.
- Why zero-duration flows produce mathematical `inf` values.

**Next up:** In **Notebook 02: Data Preprocessing Pipeline**, we will clean infinite values, drop identifier columns to prevent data leakage, clip extreme outliers, split into train/val/test, scale features, and apply SMOTE to rebalance our training data!
